In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parents[1]   # move up from notebooks/
sys.path.insert(0, str(PROJECT_ROOT))
from pathlib import Path
from proteins.data.datasets import ESMCSingleDS
from proteins.data.utils import pad_collate_fn
from proteins.models.model import SequenceActiveSiteHead
import torch
from proteins.training.losses import BinaryFocalLoss
from torch.utils.data import DataLoader
import torch.utils.benchmark as benchmark
from copy import deepcopy
from multiprocessing import cpu_count

data_name = 'IEDB_Jespersen'
model_name = 'esmc_300m'
base_data_dir = Path.cwd().parents[1]/'proteins'  / 'data'/ 'data_files'

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')
dataset=ESMCSingleDS(data_name, model_name, save_dir=base_data_dir)

model1 = SequenceActiveSiteHead(dataset.embed_dim, layers=5, hidden_dim=512).to(device)
model2 = deepcopy(model1).to(device)
optim1 = torch.optim.AdamW(model1.parameters(), lr=1e-3)
optim2 = torch.optim.AdamW(model2.parameters(), lr=1e-3)
num_workers = cpu_count() // 2 if device.type == 'cuda' else 0
prefetch_factor = 2 if device.type == 'cuda' else None

class SwitchableCollate:
    def __init__(self, pad_collate, zip_collate):
        self.pad_collate = pad_collate
        self.zip_collate = zip_collate
        self.use_pad = True   # ← toggle this to switch mode

    def __call__(self, batch):
        if self.use_pad:
            return self.pad_collate(batch)
        else:
            return self.zip_collate(batch)   # or lambda batch: zip(*batch)

dataloader1 = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=pad_collate_fn, pin_memory=torch.cuda.is_available(),                                         num_workers=num_workers,
                                  prefetch_factor=prefetch_factor,
                                  persistent_workers=device.type == 'cuda')
dataloader2 = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=lambda batch: zip(*batch), pin_memory=torch.cuda.is_available(),                              num_workers=num_workers,
                                  prefetch_factor=prefetch_factor,
                                  persistent_workers=device.type == 'cuda')


Dropped 4 sequences over len 5000


In [2]:
def forward1(model, optim, dataloader, device):
    criterion = BinaryFocalLoss(reduction='mean')
    for embeds, labels, mask in dataloader:
        embeds, labels, mask = embeds.to(device), labels.to(device), mask.to(device)
        logits = model(embeds)
        loss = criterion(logits, labels, mask=mask)
        optim.zero_grad()
        loss.backward()
        optim.step()

def forward2(model, optim, dataloader, device):
    criterion = BinaryFocalLoss(reduction='mean')
    for embeds, labels in dataloader:
        total_loss = torch.tensor(0.).to(device)
        for embed, label in zip(embeds, labels):
            embed, label = embed.to(device).unsqueeze(0), label.to(device).unsqueeze(0)
            logit = model(embed)
            loss = criterion(logit, label)
            total_loss += loss
        optim.zero_grad()
        total_loss.backward()
        optim.step()

In [11]:
forward1(model1, optim1, dataloader1, device)

RuntimeError: MPS backend out of memory (MPS allocated: 8.22 GiB, other allocations: 552.89 MiB, max allowed: 9.07 GiB). Tried to allocate 546.25 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [3]:
timer1 = benchmark.Timer(
    stmt='forward1(model, optim, dataloader, device)',
    setup='from __main__ import forward1',
    globals={'model': model1, 'optim': optim1, 'dataloader': dataloader1, 'device': device},
    description=f'base'
)
timer2 = benchmark.Timer(
    stmt='forward2(model, optim, dataloader, device)',
    setup='from __main__ import forward2',
    globals={'model': model2, 'optim': optim2, 'dataloader': dataloader2, 'device': device},
    description=f'split'
)
measurement1 = timer1.timeit(number=1)
measurement2 = timer2.timeit(number=1)
print(measurement1)
print(measurement2)

RuntimeError: MPS backend out of memory (MPS allocated: 8.00 GiB, other allocations: 760.70 MiB, max allowed: 9.07 GiB). Tried to allocate 753.25 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).